In [ ]:
# from transformers import AutoModelForCausalLM, AutoToken  izer
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForImageTextToText, AutoProcessor
import os 
import shutil

output_path = "./qwen3-vl-lora-merged-qwen"
shutil.rmtree(output_path)

# default: Load the model on the available device(s)
model_id = "Qwen/Qwen3-VL-8B-Instruct"
model = AutoModelForImageTextToText.from_pretrained(
    model_id, dtype="auto", device_map='cuda:2'
)  # device_map="auto"


lora_model_id = 'path/to//VLM-R1-Qwen3/src/open-r1-multimodal/output/your_model/checkpoint-600' 

processor = AutoProcessor.from_pretrained(model_id)

# 3. Load base model
# 4. Load and merge LoRA
model = PeftModel.from_pretrained(model, lora_model_id)
model = model.merge_and_unload() 


if not os.path.exists(output_path):
    os.mkdir(output_path)    
# else:
#     shutil.rmtree(output_path)
    
model.save_pretrained(output_path)
processor.save_pretrained(output_path)

print(f"✅ LoRA-merged model saved to: {output_path}")


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "your_repo"  

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,           
    exist_ok=True           
)

api.upload_folder(
    repo_id=repo_id,
    folder_path="./qwen3-vl-lora-merged-qwen",
    repo_type="model",
    commit_message="Upload merged LoRA checkpoint"
)
